# Analiza cen i źródeł energii w Polsce i Europie (2015–2025)

**Projekt zaliczeniowy** — Analiza i wizualizacja danych (Pandas)

# Transformacja Energetyczna Polski (2015–2024)
## Raport Analityczny na bazie platformy ENTSO-E

| Metadane Projektu | Opis |
| :--- | :--- |
| **Uczelnia** | Uniwersytet WSB Merito Chorzów |
| **Kierunek / Przedmiot** | Analiza i wizualizacja danych – Pandas, DataFrame |
| **Zespół** | Dawid Hetmańczyk (126674), Bartosz Bugla (180737) |
| **Zakres czasowy** | 2015-01-01 do 2025-12-31 (10 lat) dla polski oraz 2020-01-01 do 2025-12-31 dla reszty kraji|
| **Dane źródłowe** | ENTSO-E Transparency Platform (API) |

---

---

## 1. Wczytanie i walidacja danych

In [1]:
from dateutil import tz
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.1)

CHART_DIR = "charts"
EXPORT_DPI = 200

EU_DATASETS = {
    "PL": { 
        "path": "datasets/energy_PL_2015-01-01_2025-12-31.csv",
        "tz": "Europe/Warsaw"
    },
    "DE": {
        "path": "datasets/energy_DE_2020-01-01_2025-12-31.csv",
        "tz": "Europe/Berlin"
    },
    "FR": {
        "path": "datasets/energy_FR_2020-01-01_2025-12-31.csv",
        "tz": "Europe/Paris"
    },
    "ES": {
        "path": "datasets/energy_ES_2020-01-01_2025-12-31.csv",
        "tz": "Europe/Madrid" # Uwaga: Wyspy Kanaryjskie mają inną strefę, ale dane ENTSO-E dla ES to zazwyczaj kontynent.
    },
    "BG": {
        "path": "datasets/energy_BG_2020-01-01_2025-12-31.csv",
        "tz": "Europe/Sofia" # Czas wschodnioeuropejski (EET/EEST)
    },
    "SE": {
        "path": "datasets/energy_SE_2020-01-01_2025-12-31.csv",
        "tz": "Europe/Stockholm"
    },
}
COLORS = {
    "Węgiel": "#5D4037",
    "Gaz": "#F57C00",
    "Wiatr": "#0288D1",
    "Słońce (PV)": "#FBC02D",
    "Inne": "#7B1FA2",
}

COUNTRY_COLORS = {
    "PL": "#E53935",
    "DE": "#1E88E5",
    "FR": "#43A047",
    "ES": "#FB8C00",
    "BG": "#8E24AA",
    "SE": "#00ACC1"
}

MIX = {
    "Węgiel": ["Fossil Hard coal", "Fossil Brown coal/Lignite"],
    "Gaz": ["Fossil Gas"],
    "Wiatr": ["Wind Onshore"],
    "Słońce (PV)": ["Solar"],
}

OTHER_COLS = [
    "Biomass", "Fossil Coal-derived gas", "Fossil Oil",
    "Hydro Pumped Storage", "Hydro Run-of-river and poundage",
    "Hydro Water Reservoir", "Other", "Other renewable",
]

OZE_COLS = [
    "Wind Onshore", "Solar", "Biomass",
    "Hydro Run-of-river and poundage", "Hydro Water Reservoir",
    "Other renewable",
]


def load_csv(path: str, tz: str = "Europe/Warsaw") -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["time"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.set_index("time").sort_index()
    df.index = df.index.tz_convert(tz)
    return df


def savefig(fig_or_name, name: str, **kwargs):
    """Save matplotlib figure to charts/ directory."""
    path = f"{CHART_DIR}/{name}.png"
    if hasattr(fig_or_name, "savefig"):
        fig_or_name.savefig(path, dpi=EXPORT_DPI, bbox_inches="tight",
                           facecolor="white", edgecolor="none", **kwargs)
    else:
        plt.savefig(path, dpi=EXPORT_DPI, bbox_inches="tight",
                    facecolor="white", edgecolor="none", **kwargs)
    print(f"  -> Zapisano: {path}")


def plot_seasonal_decompose(series, title, period, model="additive", highlight_covid=False, height=800, width=800):
    decomp = seasonal_decompose(series, model=model, period=period)
    
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=(
            "<b>Obserwowane zapotrzebowanie</b>",
            "<b>Trend</b>",
            f"<b>Sezonowość ({'roczna' if period == 365 else 'tygodniowa'})</b>",
            "<b>Reszty (anomalie)</b>"
        )
    )
    
    components = [
        (decomp.observed, "#42A5F5", 1, "lines"),
        (decomp.trend, "#66BB6A", 2, "lines"),
        (decomp.seasonal, "#FFA726", 3, "lines"),
        (decomp.resid, "#EF5350", 4, "markers"),
    ]
    
    for data, color, row, mode in components:
        if mode == "markers":
            fig.add_trace(
                go.Scatter(
                    x=data.index,
                    y=data.values,
                    mode='markers',
                    marker=dict(color=color, size=3),
                    showlegend=False
                ),
                row=row, col=1
            )
        else:
            fig.add_trace(
                go.Scatter(
                    x=data.index,
                    y=data.values,
                    mode='lines',
                    line=dict(color=color, width=1),
                    showlegend=False
                ),
                row=row, col=1
            )
        fig.update_yaxes(title_text="MW", row=row, col=1)
        
    for annotation in fig['layout']['annotations']:
        annotation['x'] = 0
        annotation['xanchor'] = 'left'
        annotation['font'] = dict(size=12, color="#2c3e50")
        
    if highlight_covid:
        tz = series.index.tz
        fig.add_vrect(
            x0=pd.Timestamp("2020-01-01", tz=tz),
            x1=pd.Timestamp("2020-12-31", tz=tz),
            fillcolor="red",
            opacity=0.15,
            line_width=0,
            row=2, col=1,
            annotation_text="COVID-19",
            annotation_position="top right"
        )
        
    fig.update_layout(
        title_text=f"<b>{title}</b>",
        title_x=0.5,
        height=height,
        width=width,
        template="plotly_white",
        margin=dict(t=100, b=50, l=60, r=40)
    )
    
    return fig


In [2]:
df_pl = load_csv(**EU_DATASETS['PL'])

print(f"Wczytano: {df_pl.shape[0]:,} wierszy x {df_pl.shape[1]} kolumn")
print(f"Zakres: {df_pl.index.min()} -> {df_pl.index.max()}")
print(f"Typ indeksu: {type(df_pl.index).__name__}")

Wczytano: 96,433 wierszy x 16 kolumn
Zakres: 2015-01-01 00:00:00+01:00 -> 2026-01-01 00:00:00+01:00
Typ indeksu: DatetimeIndex


In [3]:

hours_per_year = df_pl.groupby(df_pl.index.year).size()
full_years = hours_per_year[hours_per_year >= 8000].index
df = df_pl[df_pl.index.year.isin(full_years)].copy()
print(f"Pelne lata: {full_years.min()}–{full_years.max()} ({len(df):,} wierszy)")

df.describe().round(1)

Pelne lata: 2015–2025 (96,432 wierszy)


,Actual Load,Biomass,Fossil Brown coal/Lignite,Fossil Coal-derived gas,Fossil Gas,Fossil Hard coal,Fossil Oil,Hydro Pumped Storage,Hydro Run-of-river and poundage,Hydro Water Reservoir,Other,Other renewable,Price DA,Solar,Wind Onshore,renewable
count,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0,96432.0
mean,19023.6,226.3,4551.7,74.9,1226.4,8221.6,202.5,112.9,172.2,20.8,84.4,8.2,112.4,678.9,1872.5,2978.9
std,3189.8,87.2,1248.7,41.9,679.8,2086.2,73.0,202.8,71.3,32.3,210.0,20.2,83.6,1762.7,1591.8,2360.4
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-133.0,0.0,8.2,145.5
25%,16392.5,194.8,3663.7,54.0,658.6,6701.4,166.4,0.0,113.8,0.0,0.0,0.0,45.3,0.0,655.7,1260.3
50%,19102.1,240.4,4635.6,65.1,1151.9,8124.0,195.8,0.7,155.3,0.0,0.0,0.0,98.0,0.0,1394.6,2242.3
75%,21506.1,265.0,5471.1,74.9,1517.4,9736.0,235.2,140.8,226.7,39.9,0.0,0.0,160.2,128.5,2659.4,3941.7
max,28303.9,451.4,7877.6,225.7,4574.1,15381.1,436.6,1608.8,373.2,325.2,753.4,76.5,1035.8,13682.3,8897.4,16192.2


In [4]:
# --- 1. GLOBALNA KONFIGURACJA STYLU ---
def setup_global_style():
    custom_template = go.layout.Template()
    custom_template.layout = {
        "font": {"family": "Arial", "size": 12, "color": "#2c3e50"},
        "title": {"x": 0.5, "font": {"size": 20, "weight": "bold"}},
        "xaxis": {"tickmode": "linear", "dtick": 1, "gridcolor": "#f2f2f2", "linecolor": "#2c3e50"},
        "yaxis": {"gridcolor": "#f2f2f2", "zerolinecolor": "#dcdcdc", "linecolor": "#2c3e50", "automargin": True},
        "width": 900,
        "height": 600,
        "hovermode": "x unified",
        "template": "plotly_white"
    }
    pio.templates["energy_project"] = custom_template
    pio.templates.default = "energy_project"

setup_global_style()

---
## 2. Ceny energii w Polsce

### 2a. Średnia roczna cena prądu (day-ahead)

In [5]:
import plotly.io as pio
import plotly.graph_objects as go
import pandas as pd

def add_market_events(fig):
    """Dodaje predefiniowane wydarzenia rynkowe do wykresu."""
    events = [
        dict(x0=2020, x1=2021, color="gray", label="Pandemia"),
        dict(x0=2022, x1=2023, color="red", label="Kryzys / Wojna")
    ]
    
    for ev in events:
        fig.add_vrect(
            x0=ev['x0'], x1=ev['x1'], 
            fillcolor=ev['color'], opacity=0.1, 
            line_width=0, layer="below",
            annotation_text=ev['label'], annotation_position="top left"
        )
    return fig

def plot_yearly_prices(df, country_name="Polsce"):
    yearly_price = df["Price DA"].groupby(df.index.year).mean()

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=yearly_price.index,
        y=yearly_price.values,
        mode='lines+markers+text',
        name='Cena DA',
        text=[f"{val:.0f}" for val in yearly_price.values],
        textposition="top center",
        line=dict(color='#00CC96', width=4),
        marker=dict(size=12, symbol="diamond"),
        textfont=dict(weight="bold")
    ))

    fig = add_market_events(fig)

    fig.update_layout(
        title_text=f"Średnia roczna cena day-ahead w {country_name}",
        yaxis_title="Cena [EUR/MWh]",
        xaxis_title="Rok"
    )
    
    return fig

# Zakładając, że df jest już gotowy:
fig = plot_yearly_prices(df, country_name="Polsce")
fig.show()

In [6]:
def plot_yearly_trend(df, column_name, country_name="Polsce", color='#00CC96'):
    """
    Generyczna funkcja do rysowania rocznych trendów dla dowolnej kolumny.
    """
    # Sprawdzenie czy kolumna istnieje
    if column_name not in df.columns:
        print(f"Błąd: Kolumna '{column_name}' nie istnieje w DataFrame.")
        return None

    # Agregacja danych
    yearly_data = df[column_name].groupby(df.index.year).mean()

    fig = go.Figure()

    # Główna seria danych
    fig.add_trace(go.Scatter(
        x=yearly_data.index,
        y=yearly_data.values,
        mode='lines+markers+text',
        name=column_name,
        text=[f"{val:.0f}" for val in yearly_data.values],
        textposition="top center",
        line=dict(color=color, width=4),
        marker=dict(size=12, symbol="diamond"),
        textfont=dict(weight="bold")
    ))

    # Nakładanie kontekstu historycznego
    fig = add_market_events(fig)

    # Dynamiczne formatowanie jednostek
    unit = "EUR/MWh" if "Price" in column_name else "MW"
    
    fig.update_layout(
        title_text=f"Średnie roczne {column_name} w {country_name}",
        yaxis_title=f"{column_name} [{unit}]",
        xaxis_title="Rok"
    )
    
    return fig

In [7]:
fig_load = plot_yearly_trend(df, "Actual Load", country_name="Polsce", color='#636EFA')
fig_load.show()
fig = plot_yearly_prices(df, country_name="Polsce")
fig.show()

### 2c. Ujemne ceny energii — efekt fotowoltaiki

Od 2023 r. na polskim rynku pojawiają się ujemne ceny day-ahead. Przyczyną jest nadprodukcja z OZE (głównie PV) w godzinach szczytu słonecznego.

In [8]:
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.graph_objects as go

# Przygotowanie danych
neg = df[df["Price DA"] < 0].copy()
neg["Rok"] = neg.index.year
neg["Godzina"] = neg.index.hour
neg_count = neg.groupby("Rok").size()

print("Liczba godzin z ujemną ceną wg roku:")
print(neg_count.to_string())
print(f"\nNajniższa cena: {df['Price DA'].min():.2f} EUR/MWh")

# Inicjalizacja wykresu z dwoma panelami (1 wiersz, 2 kolumny)
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Liczba godzin z ujemną ceną", "Pora dnia ujemnych cen")
)

# Panel 1: ile godzin ujemnych per rok (Bar)
fig.add_trace(
    go.Bar(
        x=neg_count.index, 
        y=neg_count.values,
        text=neg_count.values,
        textposition='outside', # odpowiednik ax.text() nad słupkiem
        name="Godziny ujemne"
    ),
    row=1, col=1
)

# Panel 2: histogram godzin dnia z ujemną ceną (Histogram)
fig.add_trace(
    go.Histogram(
        x=neg["Godzina"],
        xbins=dict(start=-0.5, end=24.5, size=1), # odpowiada bins=range(25) w matplotlib
        name="Rozkład"
    ),
    row=1, col=2
)

# Konfiguracja osi
fig.update_xaxes(title_text="Rok", type='category', row=1, col=1) # type='category' wymusi poprawne wyświetlanie lat
fig.update_yaxes(title_text="Godziny", row=1, col=1)

fig.update_xaxes(title_text="Godzina", tickmode='linear', tick0=0, dtick=2, row=1, col=2)
fig.update_yaxes(title_text="Liczba wystąpień", row=1, col=2)

# Adnotacja dla Szczytu PV (prawy wykres)
fig.add_annotation(
    x=12.5,
    y=0.85, 
    yref="y2 domain", # Ustawia pozycję na podstawie % wysokości prawego wykresu (85%)
    text="Szczyt PV<br>(10–15)",
    showarrow=False,
    row=1, col=2
)

# Ogólny układ
fig.update_layout(
    title_text="Ujemne ceny energii — efekt nadprodukcji z fotowoltaiki",
    showlegend=False,
    barmode='group'
)

fig.show()


Liczba godzin z ujemną ceną wg roku:
Rok
2023     43
2024    197
2025    310

Najniższa cena: -132.95 EUR/MWh


---
## 3. Czy Polska jest eko? Co z tym węglem?

### 3a. Ewolucja miksu energetycznego (2015–2025)

In [9]:
import plotly.graph_objects as go

yearly = df.resample("YE").sum(numeric_only=True) / 1e6
yearly.index = yearly.index.year

mix_twh = pd.DataFrame({name: yearly[cols].sum(axis=1) for name, cols in MIX.items() if any(c in yearly for c in cols)})
mix_twh["Inne"] = yearly[[c for c in OTHER_COLS if c in yearly]].sum(axis=1)

mix_pct = mix_twh.div(mix_twh.sum(axis=1), axis=0) * 100

fig = go.Figure()

for col in mix_pct.columns:
    fig.add_trace(go.Bar(
        x=mix_pct.index,
        y=mix_pct[col],
        name=col,
        marker_color=COLORS.get(col),
        text=mix_pct[col].apply(lambda x: f"{x:.1f}%" if x > 3 else ""),

        textposition='inside',
        insidetextanchor='middle',
        textfont=dict(color="white" if col in ["Węgiel", "Gaz"] else "black", size=10)
    ))

fig.update_layout(
    title_text="Ewolucja miksu energetycznego Polski (2015–2025)",
    barmode='stack',
    yaxis_title="Udział [%]",
    xaxis_title="Rok",
    yaxis=dict(ticksuffix="%", range=[0, 105]),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.2,
        xanchor="center",
        x=0.5
    )
)

fig.show()

### 3b. Spadek produkcji z węgla — trend

In [10]:
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.graph_objects as go

# --- 1. Przygotowanie danych ---
coal_yearly = mix_twh["Węgiel"]
coal_cols = MIX["Węgiel"]
monthly_data = {year: df[df.index.year == year][coal_cols].sum(axis=1).resample("ME").sum() / 1e6 
                for year in [2015, 2024]}

fig = make_subplots(
    rows=2, cols=1, 
    subplot_titles=("Roczna produkcja energii z węgla (Trend długoterminowy)", 
                    "Porównanie sezonowości węgla: 2015 vs 2024"),
    vertical_spacing=0.15 # Odstęp między wykresami
)

fig.add_trace(
    go.Scatter(
        x=coal_yearly.index, 
        y=coal_yearly.values,
        mode='lines+markers+text',
        name='Produkcja roczna',
        text=[f"{val:.1f}" for val in coal_yearly.values],
        textposition="top center",
        line=dict(color='#5D4037', width=4),
        fill='tozeroy',
        fillcolor='rgba(93, 64, 55, 0.15)'
    ),
    row=1, col=1
)

colors = {2015: "#8D6E63", 2024: "#2E7D32"}
markers = {2015: "square", 2024: "circle"}

for year in [2015, 2024]:
    fig.add_trace(
        go.Scatter(
            x=list(range(1, 13)), 
            y=monthly_data[year].values,
            mode='lines+markers',
            name=f"Rok {year}",
            line=dict(color=colors[year], width=3),
            marker=dict(symbol=markers[year], size=10)
        ),
        row=2, col=1
    )

# --- 3. Konfiguracja Layoutu ---
fig.update_layout(
    height=900, # Zwiększona wysokość dla układu pionowego
    width=750,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=-0.1, xanchor="center", x=0.5)
)

# Ustawienia osi dla wykresu górnego
fig.update_xaxes(title_text="Rok", row=1, col=1)
fig.update_yaxes(title_text="TWh", row=1, col=1)

# Ustawienia osi dla wykresu dolnego
fig.update_xaxes(title_text="Miesiąc", tickmode='linear', dtick=1, range=[0.5, 12.5], row=2, col=1)
fig.update_yaxes(title_text="TWh", row=2, col=1)

# Dodanie kontekstu historycznego do górnego wykresu
for ev in [{"x0": 2020, "x1": 2021, "label": "Pandemia", "color": "gray"}, 
           {"x0": 2022, "x1": 2023, "label": "Kryzys", "color": "red"}]:
    fig.add_vrect(
        x0=ev['x0'], x1=ev['x1'], 
        fillcolor=ev['color'], opacity=0.1, line_width=0,
        annotation_text=ev['label'], annotation_position="top left",
        row=1, col=1
    )

fig.show()

---
## 4. Polska na tle Unii Europejskiej

Porównanie średnich cen i zużycia energii w wybranych krajach za okres 2020–2024.

In [11]:
COUNTRY_COLORS = {
    "PL": "#E53935",
    "DE": "#1E88E5",
    "FR": "#43A047",
    "ES": "#FB8C00",
    "BG": "#8E24AA",
    "SE": "#00ACC1"
}

def get_yearly_oze_data():
    yearly_data = {}
    for country, config in EU_DATASETS.items():
        d = load_csv(**config)
        y = d.loc["2020":"2024"]
        yearly_data[country] = y.groupby(y.index.year).apply(lambda x: pd.Series({
            "OZE_share": (x["renewable"].sum() / x["Actual Load"].sum() * 100) if "renewable" in x.columns and "Actual Load" in x.columns else np.nan,
            "OZE_twh": (x["renewable"].sum() / 1e6) if "renewable" in x.columns else np.nan,
            "load_twh": (x["Actual Load"].sum() / 1e6) if "Actual Load" in x.columns else np.nan,
            "price_da": (x["Price DA"].mean()) if "Price DA" in x.columns else np.nan
        }))
    return yearly_data

# Wczytanie i przygotowanie danych (jednokrotne wykonanie)
yearly_oze_data = get_yearly_oze_data()

def plot_europe_with_metric(metric: str, title: str, yaxis_title: str, hover_format: str, filename: str):
    fig = go.Figure()
    for country, data in yearly_oze_data.items():
        # Pomijaj rysowanie linii, jeśli cała seria jest pusta (NaN)
        if data[metric].isna().all():
            continue
        fig.add_trace(
            go.Scatter(
                x=data.index,
                y=data[metric],
                mode="lines+markers",
                name=country,
                line=dict(color=COUNTRY_COLORS.get(country, "#757575"), width=3),
                marker=dict(size=8),
                hovertemplate=f"<b>{country}</b><br>Rok: %{{x}}<br>{title}: %{{y:{hover_format}}}<extra></extra>"
            )
        )
    fig.update_layout(
        title=dict(text=title, x=0.5),
        xaxis=dict(title="Rok", dtick=1, tickmode="linear"),
        yaxis=dict(title=yaxis_title),
        height=450,
        width=700,
        legend=dict(orientation="h", yanchor="bottom", y=-0.35, xanchor="center", x=0.5)
    )
    try:
        fig.write_html(f"charts/{filename}.html")
    except Exception:
        pass
    fig.show()

In [12]:
plot_europe_with_metric(
    metric="load_twh",
    title="Roczne zapotrzebowanie na energię (2020–2024)",
    yaxis_title="TWh",
    hover_format=".1f} TWh",
    filename="07c_zapotrzebowanie_ue"
)

In [13]:
plot_europe_with_metric(
    metric="price_da",
    title="Średnia roczna cena DA (2020–2024)",
    yaxis_title="EUR/MWh",
    hover_format=".1f} EUR/MWh",
    filename="08_ceny_ue_roczne"
)

In [14]:
plot_europe_with_metric(
    metric="OZE_share",
    title="Udział OZE w całkowitej generacji (2020–2024)",
    yaxis_title="%",
    hover_format=".1f}%",
    filename="07a_udzial_oze_ue"
)

In [15]:
plot_europe_with_metric(
    metric="OZE_twh",
    title="Roczna produkcja energii z OZE (2020–2024)",
    yaxis_title="TWh",
    hover_format=".1f} TWh",
    filename="07b_produkcja_oze_ue"
)

In [16]:
plot_europe_with_metric(
    metric="load_twh",
    title="Roczne zapotrzebowanie na energię (2020–2024)",
    yaxis_title="TWh",
    hover_format=".1f} TWh",
    filename="07c_zapotrzebowanie_ue"
)

In [17]:
plot_europe_with_metric(
    metric="price_da",
    title="Średnia roczna cena DA (2020–2024)",
    yaxis_title="EUR/MWh",
    hover_format=".1f} EUR/MWh",
    filename="08_ceny_ue_roczne"
)

---
## 5. Dekompozycja szeregu czasowego (zapotrzebowanie)

Addytywna dekompozycja dziennego zapotrzebowania: trend, sezonowość roczna, reszty.

In [18]:
load_daily = df["Actual Load"].resample("D").mean().loc["2015":"2024"]

result_adf = adfuller(load_daily.dropna())
print(f"Test ADF (stacjonarność): stat={result_adf[0]:.2f}, p={result_adf[1]:.2e}")

Test ADF (stacjonarność): stat=-5.19, p=9.36e-06


In [19]:
fig = plot_seasonal_decompose(
    series=load_daily,
    title="Dekompozycja szeregu czasowego — dzienne zapotrzebowanie (2015–2024)",
    period=365,
    highlight_covid=True,
    height=900
)
fig.show()

### Wartości odstające - święta i anomalie na zapotrzebowaniu

Polskie święta państwowe powodują spadek zapotrzebowania podobny do weekendów.

In [20]:
# Dekompozycja Szeregu Czasowego (Na przykładzie 2023 roku)
df_2023 = df_pl.loc["2020"].resample("D").mean(numeric_only=True).dropna(subset=["Actual Load"])
fig = plot_seasonal_decompose(
    series=df_2023["Actual Load"],
    title="Dekompozycja zapotrzebowania na prąd (2020)",
    period=7,
    highlight_covid=False,
    height=800
)
fig.show()

---
## 6. Weekend vs dzień roboczy — analiza statystyczna

### 6a. Wizualizacja różnicy

In [21]:
load_h = df[["Actual Load"]].copy()
load_h["is_weekend"] = load_h.index.dayofweek >= 5
load_h["Godzina"] = load_h.index.hour
load_h["Typ"] = np.where(load_h["is_weekend"], "Weekend", "Dzień roboczy")

mean_wd = load_h.loc[~load_h["is_weekend"], "Actual Load"].mean()
mean_we = load_h.loc[load_h["is_weekend"], "Actual Load"].mean()
pct_diff = (mean_we - mean_wd) / mean_wd * 100
print(f"Średnie zapotrzebowanie:")
print(f"  Dzień roboczy: {mean_wd:,.0f} MW")
print(f"  Weekend:       {mean_we:,.0f} MW")
print(f"  Różnica:       {pct_diff:+.1f}%")

# Wykres 1: Rozkład zapotrzebowania (Box Plot)
data_wd = load_h.loc[~load_h["is_weekend"], "Actual Load"].sample(5000, random_state=42)
data_we = load_h.loc[load_h["is_weekend"], "Actual Load"].sample(5000, random_state=42)

fig1 = go.Figure()
fig1.add_trace(go.Box(y=data_wd, name="Dzień roboczy", showlegend=False))
fig1.add_trace(go.Box(y=data_we, name="Weekend", showlegend=False))

fig1.update_layout(
    title_text="Rozkład zapotrzebowania na energię: Dzień roboczy vs Weekend",
    height=500
)
fig1.update_yaxes(title_text="MW")

fig1.show()


Średnie zapotrzebowanie:
  Dzień roboczy: 19,860 MW
  Weekend:       16,932 MW
  Różnica:       -14.7%


In [22]:
profile = load_h.groupby(["Typ", "Godzina"])["Actual Load"].mean().reset_index()

fig2 = go.Figure()
for typ in ["Dzień roboczy", "Weekend"]:
    sub = profile[profile["Typ"] == typ]
    fig2.add_trace(
        go.Scatter(
            x=sub["Godzina"],
            y=sub["Actual Load"],
            mode="lines+markers",
            name=typ
        )
)

fig2.update_layout(
    title_text="Profil dobowy zapotrzebowania na energię",
    height=500
)
fig2.update_xaxes(title_text="Godzina", tickvals=list(range(0, 24, 2)), dtick=2)
fig2.update_yaxes(title_text="MW")

fig2.show()


### 6b. Testy statystyczne

Weryfikujemy hipotezę: *"Zapotrzebowanie w weekend jest istotnie niższe niż w dni robocze"*

- **H₀:** brak różnicy w średnich
- **H₁:** średnia weekendowa < średnia robocza

Stosujemy: Mann-Whitney U (dane nie są normalne — potwierdzone wcześniej testem Shapiro-Wilka).

In [23]:
workday = load_h.loc[~load_h["is_weekend"], "Actual Load"].dropna()
weekend = load_h.loc[load_h["is_weekend"], "Actual Load"].dropna()

# 1. Test normalności Shapiro-Wilka
for name, s in [("Dzień roboczy", workday), ("Weekend", weekend)]:
    sample = s.sample(min(5000, len(s)), random_state=42)
    stat_sw, p_sw = stats.shapiro(sample)
    print(f"{name}: n={len(s):,}, skew={s.skew():.2f}, kurtosis={s.kurtosis():.2f}")
    print(f"  Shapiro-Wilk: W={stat_sw:.4f}, p={p_sw:.2e} -> {'NIE normalny' if p_sw < 0.05 else 'normalny'}")


Dzień roboczy: n=68,880, skew=-0.28, kurtosis=-0.72
  Shapiro-Wilk: W=0.9783, p=5.54e-27 -> NIE normalny
Weekend: n=27,552, skew=0.19, kurtosis=-0.38
  Shapiro-Wilk: W=0.9945, p=6.84e-13 -> NIE normalny


In [24]:
# 2. Mann-Whitney U test (nieparametryczny)
u_stat, p_mw = stats.mannwhitneyu(workday, weekend, alternative="greater")
print(f"Mann-Whitney U test (H1: roboczy > weekend):")
print(f"  U = {u_stat:,.0f}, p = {p_mw:.2e}")
print(f"  -> {'ODRZUCAMY H0' if p_mw < 0.05 else 'Brak podstaw do odrzucenia H0'} (α=0.05)")


Mann-Whitney U test (H1: roboczy > weekend):
  U = 1,456,277,098, p = 0.00e+00
  -> ODRZUCAMY H0 (α=0.05)


In [25]:
# 3. Welch t-test (parametryczny, uzupełniająco)
t_stat, p_t = stats.ttest_ind(workday, weekend, alternative="greater", equal_var=False)
print(f"Welch t-test:")
print(f"  t = {t_stat:.2f}, p = {p_t:.2e}")


Welch t-test:
  t = 161.28, p = 0.00e+00


In [26]:
# 4. Wielkość efektu (Cohen's d)
d = (workday.mean() - weekend.mean()) / np.sqrt((workday.std()**2 + weekend.std()**2) / 2)
print(f"Cohen's d = {d:.3f} ({'duży' if abs(d) > 0.8 else 'średni' if abs(d) > 0.5 else 'mały'} efekt)")


Cohen's d = 1.072 (duży efekt)


In [27]:
import plotly.express as px

# 1. Przygotowanie danych (tak samo jak w Matplotlib)
summary = pd.DataFrame({
    "Typ": ["Dzień roboczy", "Weekend"],
    "Średnia [MW]": [workday.mean(), weekend.mean()],
    "SE": [workday.std() / np.sqrt(len(workday)), weekend.std() / np.sqrt(len(weekend))],
})

# 2. Obliczenie odchylenia dla przedziału ufności 95%
summary["Error_95CI"] = summary["SE"] * 1.96

# 3. Rysowanie wykresu z Plotly Express
fig = px.bar(
    summary, 
    x="Typ", 
    y="Średnia [MW]", 
    error_y="Error_95CI",
    text="Średnia [MW]", # Automatycznie dodaje etykiety tekstu
    title=f"Średnie zapotrzebowanie z 95% CI<br>Mann-Whitney p = {p_mw:.2e}, Cohen's d = {d:.2f}"
)

# 4. Kosmetyka dla formatowania liczb na słupkach (np. 20,000 zamiast 20000.5)
fig.update_traces(texttemplate='%{text:,.0f}', textposition='outside')
fig.update_layout(yaxis_title="MW")

fig.show()


---
## 7. Analiza rozkładów i test normalności

### Miary statystyczne i test normalności

Krótkie wprowadzenie:
- **Skośność**: mierzy asymetrię rozkładu ($>0$ to długi prawy ogon, $<0$ lewy).
- **Kurtoza**: mierzy grubość ogonów (wartości ekstremalne) względem rozkładu normalnego.
- **Test Shapiro-Wilka**: sprawdza, czy dane pochodzą z rozkładu normalnego ($p < 0.05$ oznacza odrzucenie tej hipotezy).


In [28]:
import plotly.express as px
from scipy import stats

# Histogram dla Ceny DA
fig_price = px.histogram(
    df, 
    x="Price DA", 
    nbins=60,
    color_discrete_sequence=["#FF7043"],
    title="Rozkład: Cena DA"
)
fig_price.update_layout(showlegend=False, template="plotly_white", xaxis=dict(tickmode="auto", dtick=None))
fig_price.show()

# Poniższa część statystyczna (Skośność, Kurtoza, Test Shapiro-Wilka)
s = df["Price DA"].dropna()
sample = s.sample(min(5000, len(s)), random_state=42)
stat, p = stats.shapiro(sample)
print(f"Price DA: skew={s.skew():.2f}, kurtosis={s.kurtosis():.2f}")
print(f"  Shapiro-Wilk: W={stat:.4f}, p={p:.2e}")
print(f"  -> {'NIE jest' if p < 0.05 else 'Jest'} normalny (α=0.05)\n")

Price DA: skew=1.21, kurtosis=3.22
  Shapiro-Wilk: W=0.9248, p=1.52e-44
  -> NIE jest normalny (α=0.05)



In [29]:
import plotly.express as px
from scipy import stats

# Histogram dla Zapotrzebowania
fig_load = px.histogram(
    df, 
    x="Actual Load", 
    nbins=60,
    color_discrete_sequence=["#42A5F5"],
    title="Rozkład: Zapotrzebowanie"
)
fig_load.update_layout(showlegend=False, template="plotly_white", xaxis=dict(tickmode="auto", dtick=None))
fig_load.show()

# Poniższa część statystyczna (Skośność, Kurtoza, Test Shapiro-Wilka)
s = df["Actual Load"].dropna()
sample = s.sample(min(5000, len(s)), random_state=42)
stat, p = stats.shapiro(sample)
print(f"Actual Load: skew={s.skew():.2f}, kurtosis={s.kurtosis():.2f}")
print(f"  Shapiro-Wilk: W={stat:.4f}, p={p:.2e}")
print(f"  -> {'NIE jest' if p < 0.05 else 'Jest'} normalny (α=0.05)\n")

Actual Load: skew=0.02, kurtosis=-0.85
  Shapiro-Wilk: W=0.9825, p=2.07e-24
  -> NIE jest normalny (α=0.05)



### Wnioski (Skośność, Kurtoza, Shapiro-Wilk)

Test Shapiro-Wilka ($p \approx 0$) odrzuca hipotezę o rozkładzie normalnym dla obu zmiennych, co jest typowe w energetyce:

- **Cena DA**: Dodatnia skośność (dolna granica ograniczona, w górę zdarzają się piki). Wysoka kurtoza (częste "grube ogony" - tzw. price spikes).
- **Zapotrzebowanie (Load)**: Mimo kształtu dzwonu, nie jest normalne z uwagi na deterministyczną sezonowość (dobową, tygodniową).

Wniosek: Modele zakładające idealną normalność będą tu wymagały odpowiednich transformacji lub innych podejść.

### Analiza Korelacji
Poniżej generujemy macierz korelacji Pearsona (oraz Spearmana), która w interaktywny sposób na mapie ciepła (heatmap) pokaże nam zależności pomiędzy zmiennymi.

In [30]:
import plotly.express as px

# Wybieramy najważniejsze zmienne do macierzy korelacji, aby była bardziej czytelna i użyteczna
cols_to_corr = {
    "Price DA": "Cena DA",
    "Actual Load": "Zapotrzebowanie",
    "Wind Onshore": "Wiatr",
    "Solar": "Słońce (PV)",
    "Fossil Hard coal": "Węgiel kamienny",
    "Fossil Brown coal/Lignite": "Węgiel brunatny",
    "Fossil Gas": "Gaz",
    "renewable": "OZE (Suma)"
}

df_corr = df[list(cols_to_corr.keys())].rename(columns=cols_to_corr)
corr_matrix = df_corr.corr(method='spearman').round(2)

# Interaktywny Heatmap przy użyciu Plotly
fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='RdBu_r', # Odwrócony gradient czerwono-niebieski
    zmin=-1, zmax=1,
    title="Macierz korelacji Pearsona (najważniejsze zmienne)"
)

fig_corr.show()

In [31]:
# Wigilia - Analiza Godzinowa (24 grudnia) vs Dzień przed (23.12)
df_24 = df[(df.index.month == 12) & (df.index.day == 24)]
df_23 = df[(df.index.month == 12) & (df.index.day == 23)]

# Obliczamy średnie po godzinach (wyciągniętych z indeksu czasowego)
df_zima_srednia = df[df.index.month == 12].groupby(df[df.index.month == 12].index.hour)["Actual Load"].mean()
srednia_24 = df_24.groupby(df_24.index.hour)["Actual Load"].mean()
srednia_23 = df_23.groupby(df_23.index.hour)["Actual Load"].mean()

fig = go.Figure()
# Szara przerywana linia - Średnia grudniowa
fig.add_trace(go.Scatter(x=df_zima_srednia.index, y=df_zima_srednia, mode="lines", 
                         line=dict(color="gray", width=3, dash='dot'), name="Średnia Grudnia"))

# Pomarańczowa linia - 23 grudnia
fig.add_trace(go.Scatter(x=srednia_23.index, y=srednia_23, mode="lines", 
                         line=dict(color="orange", width=4), name="Dzień przed (23.12)"))

# Karmazynowa linia - 24 grudnia (Wigilia)
fig.add_trace(go.Scatter(x=srednia_24.index, y=srednia_24, mode="lines", 
                         line=dict(color="crimson", width=4), name="Wigilia (24.12)"))

fig.update_layout(
    title="Profil zapotrzebowania: Wigilia vs 23.12 vs Średnia dla Grudnia", 
    xaxis_title="Godzina w ciągu doby", 
    yaxis_title="MW", 
    template="plotly_white", # Dostosowano do stylistyki z _analysis.ipynb
    hovermode="x unified",
    xaxis=dict(tickmode="linear", dtick=1)
)
fig.show()


---
## 8. Podsumowanie analiz i wnioski

1. **Mix energetyczny:** udział węgla spada (2015: ~83% -> 2024: ~55%), rośnie wiatr i PV — transformacja postępuje, ale węgiel nadal dominuje.
2. **Ceny:** gwałtowne zmiany wynikają z integracji rynkowej (XI 2019), pandemii i kryzysu energetycznego (2021–22).
3. **Ujemne ceny:** od 2023 r. pojawiają się ujemne ceny DA — głównie w godzinach 10–15 (szczyt PV), co obniża średnią roczną.
4. **OZE a ceny:** ujemna korelacja OZE% <-> cena wzmacnia się od ~2022 — im więcej OZE, tym silniejszy efekt obniżki cen.
5. **Zapotrzebowanie:** istotnie niższe w weekend (~−17%), wyraźny profil dobowy — potwierdzone testem Mann-Whitney U.
6. **Polska vs UE:** wyższe ceny historycznie vs DE, zbliżanie się w okresie 2020–2024.
7. **Dekompozycja:** trend wyraźnie pokazuje pandemię 2020, sezonowość roczna dobrze uchwycona, reszty wskazują na święta.

---